Assignmnent:
1. Load 5mn bars for US stock
2. Plot terminal distributions from time t to EOD
3. Compare with risk-neutral probabilities
4. Backtest 0DTE Condor Strategy

In [1]:
import pandas as pd
import numpy as np
import sys
import plotly.express as px
import plotly.graph_objects as go
from time import time
from tqdm import tqdm
from itertools import product
import pytz
from datetime import datetime as dt

sys.path.append('../../')

In [2]:
PERIOD = 5    # minutes
DAILY_PERIODS = 6.5 * 60 / PERIOD    # number of trading periods in a day
ANNUAL_FACTOR = 252 * DAILY_PERIODS
STOCK = 'SPY'
TZ_EXCHANGE = 'America/Chicago'

### Load Intraday Bar Data

In [3]:
# load OHLCV data from market_data/intraday/
ohlcv = pd.read_csv(f'../../market_data/intraday/{STOCK}_{PERIOD}mn.csv')
ohlcv = ohlcv.set_index('Date')
ohlcv.index = pd.to_datetime(ohlcv.index)
# convert UTC data to local time: Chicago time
ohlcv.index = ohlcv.index.tz_convert(TZ_EXCHANGE)
ohlcv.tail()

,Open,High,Low,Close,Volume
Date,,,,,
2026-03-17 14:35:00-05:00,671.00,671.14,670.94,670.99,11439.0
2026-03-17 14:40:00-05:00,670.99,671.15,670.75,670.86,18665.0
2026-03-17 14:45:00-05:00,670.87,671.21,670.85,671.18,23100.0
2026-03-17 14:50:00-05:00,671.18,671.28,670.58,671.21,44420.0
2026-03-17 14:55:00-05:00,671.20,671.20,670.47,670.73,73989.0


In [5]:
# keep only Close prices
# add each row's day expiry timestamp and its corresponding close

# close-only frame
df_price = ohlcv[['Close']].copy()

# last Date index for each trading day
expiry_by_day = df_price.index.to_series().groupby(df_price.index.normalize()).max()

# add expiry timestamp and close at expiry
df_price['Expiry'] = df_price.index.normalize().map(expiry_by_day)
df_price['Close_At_Expiry'] = df_price['Expiry'].map(df_price['Close'])

In [12]:
# add a column for the time to expiry
df_price['time_to_expiry'] = (df_price['Expiry'] - df_price.index).dt.total_seconds() / 3600
df_price.tail()

,Close,Expiry,Close_At_Expiry,time_to_expiry
Date,,,,
2026-03-17 14:35:00-05:00,670.99,2026-03-17 14:55:00-05:00,670.73,0.333333
2026-03-17 14:40:00-05:00,670.86,2026-03-17 14:55:00-05:00,670.73,0.250000
2026-03-17 14:45:00-05:00,671.18,2026-03-17 14:55:00-05:00,670.73,0.166667
2026-03-17 14:50:00-05:00,671.21,2026-03-17 14:55:00-05:00,670.73,0.083333
2026-03-17 14:55:00-05:00,670.73,2026-03-17 14:55:00-05:00,670.73,0.000000
